In [0]:
# ETL - Squad 3 - ecommerce_rastreamento_entregas
# Camada Gold
# Notebook: problemas mensais em entregas
# Fluxo inicial: Silver Delta -> análise da regra


In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
SILVER_TABLE = "ecommerce_rastreamento_entregas"
GOLD_TABLE = "gold_ecommerce_rastreamento_entregas_problemas_mensal"

SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

GOLD_KEY_COLUMNS = [
    "ano_evento",
    "mes_evento",
    "id_transportadora"
]

SQL_FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"
SQL_STAGING_TABLE = f"{TARGET_SCHEMA}.stg_{GOLD_TABLE}"

print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("GOLD_KEY_COLUMNS:", GOLD_KEY_COLUMNS)
print("SQL_FINAL_TABLE:", SQL_FINAL_TABLE)
print("SQL_STAGING_TABLE:", SQL_STAGING_TABLE)

In [0]:
# leitura silver

adls_options = get_adls_options()

df_silver = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

df_silver.printSchema()

display(df_silver.limit(10))

In [0]:
# validação de colunas

required_columns = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "id_transportadora",
    "status_entrega",
    "dt_evento",
    "observacao",
    "ano",
    "mes"
]

validate_required_columns(df_silver, required_columns)

print("Colunas obrigatórias validadas com sucesso.")

display(
    df_silver
    .groupBy("observacao")
    .count()
    .orderBy("count", ascending=False)
)

In [0]:
# imports para KPI

from pyspark.sql.functions import (
    col,
    trim,
    lower,
    year,
    month,
    lit,
    when,
    countDistinct,
    sum as spark_sum,
    max as spark_max,
    round as spark_round,
    current_timestamp,
    to_date,
    concat_ws,
    lpad
)

In [0]:
# marcação de registros com problemas na malha

TEXTO_PROBLEMA = "problemas na malha"

df_base_problemas = (
    df_silver
    .filter(col("id_pedido_ecommerce").isNotNull())
    .filter(col("id_transportadora").isNotNull())
    .filter(col("dt_evento").isNotNull())
    .withColumn("ano_evento", year(col("dt_evento")))
    .withColumn("mes_evento", month(col("dt_evento")))
    .withColumn("observacao_tratada", lower(trim(col("observacao"))))
    .withColumn(
        "fl_problema",
        when(col("observacao_tratada").contains(TEXTO_PROBLEMA), 1).otherwise(0)
    )
)
display(
    df_base_problemas
    .groupBy("observacao_tratada", "fl_problema")
    .count()
    .orderBy("fl_problema", "count", ascending=False)
)

In [0]:
# consolidação por pedido e mês 

df_pedido_mes = (
    df_base_problemas
    .groupBy(
        "ano_evento",
        "mes_evento",
        "id_transportadora",
        "id_pedido_ecommerce"
    )
    .agg(
        spark_max("fl_problema").alias("fl_pedido_com_problema")
    )
)

display(
    df_pedido_mes
    .groupBy("ano_evento", "mes_evento", "id_transportadora", "fl_pedido_com_problema")
    .count()
    .orderBy("ano_evento", "mes_evento", "id_transportadora", "fl_pedido_com_problema")
)

In [0]:
df_gold = (
    df_pedido_mes
    .groupBy(
        "ano_evento",
        "mes_evento",
        "id_transportadora"
    )
    .agg(
        countDistinct("id_pedido_ecommerce").alias("qtd_pedidos_mes"),
        spark_sum("fl_pedido_com_problema").alias("qtd_pedidos_com_problema")
    )
    .withColumn(
        "percentual_pedidos_com_problema",
        spark_round(
            (col("qtd_pedidos_com_problema") / col("qtd_pedidos_mes")) * 100,
            2
        )
    )
    .withColumn(
        "data_referencia",
        to_date(
            concat_ws(
                "-",
                col("ano_evento"),
                lpad(col("mes_evento"), 2, "0"),
                lit("01")
            )
        )
    )
    .withColumn("gold_processed_at", current_timestamp())
    .select(
        "ano_evento",
        "mes_evento",
        "data_referencia",
        "id_transportadora",
        "qtd_pedidos_mes",
        "qtd_pedidos_com_problema",
        "percentual_pedidos_com_problema",
        "gold_processed_at"
    )
    .orderBy("ano_evento", "mes_evento", "id_transportadora")
)

display(df_gold)

In [0]:
# validação se a gold já possui registros

total_gold = df_gold.count()

print("Total de linhas na Gold em memória:", total_gold)

if total_gold == 0:
    raise Exception("A Gold em memória está vazia.")

print("Validação OK: Gold gerada com registros.")

In [0]:
# validar duplicidade da chave lógica

from pyspark.sql.functions import count

df_gold_duplicadas = (
    df_gold
    .groupBy(GOLD_KEY_COLUMNS)
    .agg(count("*").alias("qtd_linhas"))
    .filter(col("qtd_linhas") > 1)
)

qtd_duplicadas = df_gold_duplicadas.count()

print("Quantidade de chaves duplicadas:", qtd_duplicadas)

if qtd_duplicadas > 0:
    display(df_gold_duplicadas)
    raise Exception("Existem chaves duplicadas na Gold.")

print("Validação OK: não existem chaves duplicadas.")

In [0]:
# validar campos obrigatórios nulos

df_gold_nulos = (
    df_gold
    .filter(
        col("ano_evento").isNull() |
        col("mes_evento").isNull() |
        col("data_referencia").isNull() |
        col("id_transportadora").isNull() |
        col("qtd_pedidos_mes").isNull() |
        col("qtd_pedidos_com_problema").isNull() |
        col("percentual_pedidos_com_problema").isNull()
    )
)

qtd_nulos = df_gold_nulos.count()

print("Quantidade de linhas com campos obrigatórios nulos:", qtd_nulos)

if qtd_nulos > 0:
    display(df_gold_nulos)
    raise Exception("Existem campos obrigatórios nulos na Gold.")

print("Validação OK: campos obrigatórios preenchidos.")

In [0]:
# validar total de pedidos

from pyspark.sql.functions import sum as spark_sum

total_pedidos_origem = (
    df_pedido_mes
    .select(
        "ano_evento",
        "mes_evento",
        "id_transportadora",
        "id_pedido_ecommerce"
    )
    .distinct()
    .count()
)

total_pedidos_gold = (
    df_gold
    .agg(spark_sum("qtd_pedidos_mes").alias("total"))
    .collect()[0]["total"]
)

print("Total de pedidos na base consolidada:", total_pedidos_origem)
print("Total de pedidos somados na Gold:", total_pedidos_gold)

if total_pedidos_origem != total_pedidos_gold:
    raise Exception("A soma de pedidos da Gold não bate com a base consolidada.")

print("Validação OK: total de pedidos conferido.")

In [0]:
# revisão de schema

df_gold.printSchema()

display(
    df_gold
    .orderBy(
        "ano_evento",
        "mes_evento",
        "id_transportadora"
    )
)

In [0]:
# verificação se gold Delta já existe

try:
    df_gold_existente = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(GOLD_PATH)
    )

    gold_existe = True

    print("Gold Delta existente encontrada.")
    print("Linhas atuais na Gold Delta:", df_gold_existente.count())

except Exception as e:
    gold_existe = False
    df_gold_existente = None

    print("Gold Delta ainda não existe. Será criada na primeira gravação.")

In [0]:
# montar upsert manual da Gold Delta

df_gold_novas_chaves = df_gold.select(GOLD_KEY_COLUMNS).distinct()

if gold_existe:
    df_gold_existente_restante = (
        df_gold_existente
        .join(
            df_gold_novas_chaves,
            on=GOLD_KEY_COLUMNS,
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_existente_restante
        .unionByName(df_gold, allowMissingColumns=True)
    )

else:
    df_gold_final = df_gold

print("Linhas que serão mantidas/gravadas na Gold Delta:", df_gold_final.count())

display(
    df_gold_final
    .orderBy(
        "ano_evento",
        "mes_evento",
        "id_transportadora"
    )
)

In [0]:
# gravar gold Delta particionada

(
    df_gold_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .options(**adls_options)
    .partitionBy("ano_evento", "mes_evento")
    .save(GOLD_PATH)
)

print("Gold Delta salva com sucesso.")

In [0]:
# leitura da gold delta gravada

df_gold_delta = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

print("Total de linhas gravadas na Gold Delta:", df_gold_delta.count())

display(
    df_gold_delta
    .orderBy(
        "ano_evento",
        "mes_evento",
        "id_transportadora"
    )
)

In [0]:
# validar quantidade gravada

total_gold_final_memoria = df_gold_final.count()
total_gold_delta = df_gold_delta.count()

print("Total esperado na Gold Delta:", total_gold_final_memoria)
print("Total lido da Gold Delta:", total_gold_delta)

if total_gold_final_memoria != total_gold_delta:
    raise Exception("A quantidade de linhas da Gold Delta não bate com o DataFrame final em memória.")

print("Validação OK: quantidade de linhas conferida.")

In [0]:
# validar duplicidade na Gold Delta

df_gold_delta_duplicadas = (
    df_gold_delta
    .groupBy(GOLD_KEY_COLUMNS)
    .agg(count("*").alias("qtd_linhas"))
    .filter(col("qtd_linhas") > 1)
)

qtd_duplicadas_delta = df_gold_delta_duplicadas.count()

print("Quantidade de chaves duplicadas na Gold Delta:", qtd_duplicadas_delta)

if qtd_duplicadas_delta > 0:
    display(df_gold_delta_duplicadas)
    raise Exception("Existem chaves duplicadas na Gold Delta.")

print("Validação OK: não existem chaves duplicadas na Gold Delta.")

In [0]:
# validar colunas obrigatórios da gold Delta

required_gold_columns = [
    "ano_evento",
    "mes_evento",
    "data_referencia",
    "id_transportadora",
    "qtd_pedidos_mes",
    "qtd_pedidos_com_problema",
    "percentual_pedidos_com_problema",
    "gold_processed_at"
]

validate_required_columns(df_gold_delta, required_gold_columns)

print("Validação OK: colunas obrigatórias da Gold Delta conferidas.")

In [0]:
# validar se as chaves da execução foram gravadas

df_chaves_nao_gravadas = (
    df_gold
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .join(
        df_gold_delta.select(GOLD_KEY_COLUMNS).distinct(),
        on=GOLD_KEY_COLUMNS,
        how="left_anti"
    )
)

qtd_chaves_nao_gravadas = df_chaves_nao_gravadas.count()

print("Quantidade de chaves da execução atual não encontradas na Gold Delta:", qtd_chaves_nao_gravadas)

if qtd_chaves_nao_gravadas > 0:
    display(df_chaves_nao_gravadas)
    raise Exception("Algumas chaves da execução atual não foram gravadas na Gold Delta.")

print("Validação OK: chaves da execução atual encontradas na Gold Delta.")

In [0]:
# validar segurança da staging

print("Schema alvo:", TARGET_SCHEMA)
print("Tabela staging:", SQL_STAGING_TABLE)

if TARGET_SCHEMA != "squad3":
    raise Exception("Schema alvo diferente de squad3. Escrita bloqueada por segurança.")

if not SQL_STAGING_TABLE.startswith("squad3.stg_"):
    raise Exception("Tabela staging fora do padrão esperado. Escrita bloqueada por segurança.")

print("Validações de segurança para staging concluídas.")

In [0]:
# prerar dataframe para o sql server staging

df_sql_staging = df_gold_delta.select(
    "ano_evento",
    "mes_evento",
    "data_referencia",
    "id_transportadora",
    "qtd_pedidos_mes",
    "qtd_pedidos_com_problema",
    "percentual_pedidos_com_problema",
    "gold_processed_at"
)

display(
    df_sql_staging
    .orderBy(
        "ano_evento",
        "mes_evento",
        "id_transportadora"
    )
)

In [0]:
# gravar staging no sql server

write_sql_table(
    df=df_sql_staging,
    table_name=SQL_STAGING_TABLE,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    mode="overwrite"
)

print("Tabela staging gravada no SQL Server com sucesso.")

In [0]:
# ler e validar staging gravada

df_sql_staging_lida = read_sql_table(
    spark=spark,
    table_name=SQL_STAGING_TABLE,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD
)

total_staging_esperado = df_sql_staging.count()
total_staging_lido = df_sql_staging_lida.count()

print("Total esperado na staging:", total_staging_esperado)
print("Total lido da staging:", total_staging_lido)

if total_staging_esperado != total_staging_lido:
    raise Exception("A quantidade de linhas da staging não bate com a Gold Delta.")

display(
    df_sql_staging_lida
    .orderBy(
        "ano_evento",
        "mes_evento",
        "id_transportadora"
    )
)

print("Staging SQL Server validada com sucesso.")

In [0]:
# validação de segurança da tabela final

print("Schema alvo:", TARGET_SCHEMA)
print("Tabela final:", SQL_FINAL_TABLE)

if TARGET_SCHEMA != "squad3":
    raise Exception("Schema alvo diferente de squad3. Escrita bloqueada por segurança.")

if not SQL_FINAL_TABLE.startswith("squad3.gold_"):
    raise Exception("Tabela final fora do padrão esperado. Escrita bloqueada por segurança.")

print("Validações de segurança para tabela final concluídas.")

In [0]:
# verificação se tabela final já existe

try:
    df_sql_final_existente = read_sql_table(
        spark=spark,
        table_name=SQL_FINAL_TABLE,
        sql_host=SQL_HOST,
        sql_database=SQL_DATABASE,
        sql_username=SQL_USERNAME,
        sql_password=SQL_PASSWORD
    )

    sql_final_existe = True

    print("Tabela final existente encontrada.")
    print("Linhas atuais na tabela final:", df_sql_final_existente.count())

except Exception as e:
    sql_final_existe = False
    df_sql_final_existente = None

    print("Tabela final ainda não existe. Será criada na primeira gravação.")

In [0]:
# montar upsert manual da tabela final

df_sql_novas_chaves = df_sql_staging_lida.select(GOLD_KEY_COLUMNS).distinct()

if sql_final_existe:
    df_sql_final_restante = (
        df_sql_final_existente
        .join(
            df_sql_novas_chaves,
            on=GOLD_KEY_COLUMNS,
            how="left_anti"
        )
    )

    df_sql_final = (
        df_sql_final_restante
        .unionByName(df_sql_staging_lida, allowMissingColumns=True)
    )

else:
    df_sql_final = df_sql_staging_lida

print("Linhas que serão gravadas na tabela final:", df_sql_final.count())

display(
    df_sql_final
    .orderBy(
        "ano_evento",
        "mes_evento",
        "id_transportadora"
    )
)

In [0]:
# gravar tabela final no SQL Server

write_sql_table(
    df=df_sql_final,
    table_name=SQL_FINAL_TABLE,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    mode="overwrite"
)

print("Tabela final gravada/atualizada no SQL Server com sucesso.")

In [0]:
# validar tabela final publicada

df_sql_final_lida = read_sql_table(
    spark=spark,
    table_name=SQL_FINAL_TABLE,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD
)

total_final_esperado = df_sql_final.count()
total_final_lido = df_sql_final_lida.count()

print("Total esperado na tabela final:", total_final_esperado)
print("Total lido da tabela final:", total_final_lido)

if total_final_esperado != total_final_lido:
    raise Exception("A quantidade de linhas da tabela final não bate com o esperado.")

df_sql_final_duplicadas = (
    df_sql_final_lida
    .groupBy(GOLD_KEY_COLUMNS)
    .agg(count("*").alias("qtd_linhas"))
    .filter(col("qtd_linhas") > 1)
)

qtd_duplicadas_final = df_sql_final_duplicadas.count()

print("Quantidade de chaves duplicadas na tabela final:", qtd_duplicadas_final)

if qtd_duplicadas_final > 0:
    display(df_sql_final_duplicadas)
    raise Exception("Existem chaves duplicadas na tabela final.")

display(
    df_sql_final_lida
    .orderBy(
        "ano_evento",
        "mes_evento",
        "id_transportadora"
    )
)

print("Tabela final SQL Server validada com sucesso.")

In [0]:
print("Resumo da execução")
print("-" * 50)

print("Notebook:")
print("03_gold_squad3_ecommerce_rastreamento_entregas_problemas_mensal")

print("\nTabela Gold Delta:")
print(GOLD_TABLE)

print("\nCaminho Gold Delta:")
print(GOLD_PATH)

print("\nTabela staging SQL Server:")
print(SQL_STAGING_TABLE)

print("\nTabela final SQL Server:")
print(SQL_FINAL_TABLE)

print("\nChave lógica:")
print(GOLD_KEY_COLUMNS)

print("\nRegra de problema considerada:")
print("observacao contém 'problemas na malha'")

print("\nTotal de linhas na Gold Delta:")
print(df_gold_delta.count())

print("\nTotal de linhas na tabela final SQL Server:")
print(df_sql_final_lida.count())

print("\nStatus:")
print("Gold mensal de problemas em entregas concluída com sucesso.")